# P06 · Contar expediciones en un horario donde el 77,5 % de las horas no son horas

> **¿Cuántas expediciones de transporte público salen por hora en un día laboral, y cómo se reparten entre buses, Metro y trenes?**

Feed GTFS estático de la Red Metropolitana de Movilidad (DTPM), 12,8 MB comprimidos y 69,7 descomprimidos. 1 098 717 filas de horarios, 26 137 expediciones, 18 486 lugares, 425 recorridos. **Licencia no verificada**: el sitio no publica condiciones de uso.

Este cuaderno corre de principio a fin en Colab. La explicación completa, con las hojas de Excel dibujadas, está en [CUADERNO.md](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/cases/gtfs_red_movilidad/CUADERNO.md).

*Generado por `scripts/build_cuadernos.py`. No editar a mano: los cambios se pierden al regenerar.*


In [ ]:
# Preparación: clona el repositorio y descarga los datos públicos.
# Los datos NO vienen del repositorio: se descargan del portal de datos
# abiertos, porque el laboratorio no redistribuye datos de terceros.
!git clone --depth 1 https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab.git lab-dtpm
%cd lab-dtpm
!pip install --quiet 'pandas>=2.2' 'numpy>=1.26' openpyxl XlsxWriter matplotlib
!python scripts/fetch_data.py gtfs-red-movilidad


## 1 · El problema

El Ministerio publica el horario completo del transporte público de Santiago en formato
GTFS: un ZIP de 12,8 MB que descomprimido pesa 69,7 MB, de los cuales 51 MB son un solo
archivo de **1 098 717 filas**.

La pregunta que llega es de las más básicas que se le pueden hacer a un horario:
*¿cuántas expediciones de transporte público salen por hora en un día laboral?* Es la
cifra que va en el primer párrafo de cualquier informe de oferta, y la que alguien usa
para decir si el servicio de la mañana es el doble que el de la tarde.

**Y hay cuatro formas de contestarla mal**, todas plausibles.

La tabla de horarios no contiene horas en el 77,5 % de sus filas: contiene
**desplazamientos desde el inicio del viaje**, porque esos recorridos no se publican con
horario fijo sino con frecuencia —«cada diez minutos entre las 6 y las 9»—. Agrupar ese
campo por su hora da 635 574 llegadas «a medianoche».

El relleno de ceros no es consistente: la misma expedición mezcla `0:00:00` y `00:00:51`,
y ordenadas como texto la primera queda **después** de la segunda.

Contar las expediciones declaradas en lugar de expandir las frecuencias subestima el
servicio por un **factor de seis**.

Y hay un cuarto problema que no existe: el 32 % de las paradas «no se usa». No es un
defecto de calidad, y reportarlo costaría más que no reportar nada.

Lo que distingue este caso de los demás es el **volumen**: aquí las decisiones de lectura
—qué columnas, qué tipos, descomprimir o no— dejan de ser estilo y pasan a decidir si el
análisis corre.


## 2 · La solución que se va a aplicar

El recorrido va de leer el archivo sin romper la memoria, a un conteo que se valida por
dos caminos independientes.

**Leer solo lo necesario** (pasos 1 a 3). Mirar el inventario del ZIP antes de abrirlo,
leer del flujo comprimido sin descomprimir a disco, y de la tabla grande pedir **cuatro
de sus ocho columnas** como texto. `dtype=str` es deliberado: queremos ver los valores
como los publicó el organismo y decidir nosotros qué es un número y qué es una hora.

**Diagnosticar antes de calcular** (pasos 4 y 5). Medir cuántas filas traen horas de
reloj y cuántas desplazamientos, y comprobar el relleno de ceros. Las dos
comprobaciones cuestan una línea y las dos cambian el resultado.

**Expandir las frecuencias** (pasos 6 y 7). Cada ventana de frecuencia representa muchas
salidas reales. La expansión se hace vectorizada con `Index.repeat`, y el fin de la
ventana es **exclusivo**: la salida de `end_time` pertenece a la ventana siguiente, y
contarla infla el total en una salida por cada una de las 14 520 ventanas.

**Contestar, y medir el error del atajo** (pasos 8 y 9). El conteo correcto por hora, y
al lado lo que habría dado cada forma equivocada, con su magnitud.

**Y el defecto que no existe** (paso 10). 5 991 de las 18 486 filas de `stops.txt` no
aparecen en la tabla de horarios. 5 989 de ellas no son paradas: son estaciones, accesos
y nodos de pasillo, distinguidos por un campo de la especificación. Las paradas
realmente huérfanas son **dos**.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cases.gtfs_red_movilidad import solution as p06
from lab.dates import parse_hms_seconds, service_day_hour
from lab.sources import zip_members

import pandas as pd

FEED = p06.FEED

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)


## 3 · Los pasos


### Paso 1 · Mirar el ZIP antes de abrirlo


Qué archivos trae, cuánto pesa cada uno descomprimido, y cuál es el grande. Decide si se puede leer todo o hay que elegir.

> ⚠️ Es el único caso del laboratorio donde Excel no alcanza. Y el límite no avisa bien: al abrir un CSV más largo, Excel carga el millón de filas que caben y **descarta el resto sin error**.


In [ ]:
zip_members(FEED)


### Paso 2 · Leer del flujo comprimido, sin descomprimir


El feed pesa 12,8 MB comprimido y 69,7 MB descomprimido. Descomprimir primero duplica el espacio sin aportar nada: `zipfile` da un flujo que `read_csv` lee directamente.


In [ ]:
tablas = p06.cargar()
{nombre: tabla.shape for nombre, tabla in tablas.items()}


### Paso 3 · Cuatro columnas de ocho, y todas como texto


De `stop_times.txt` se usan cuatro de sus ocho columnas. Y `dtype=str` es deliberado: queremos ver los valores como los publicó el organismo y decidir nosotros qué es un número y qué es una hora.

> ⚠️ Dejar que pandas infiera los tipos convierte las horas a algo **antes** de que las validemos, y las de 24:00 o más se vuelven nulas o texto según la fila. El diagnóstico del paso siguiente ya no se podría hacer.


In [ ]:
horarios = tablas['stop_times']
{'filas': len(horarios), 'columnas': list(horarios.columns),
 'memoria_MB': round(horarios.memory_usage(deep=True).sum() / 1e6, 1)}


### Paso 4 · El 77,5 % de las «horas» no son horas


La comprobación que decide todo el resto. Una expedición con frecuencia declarada publica en `arrival_time` el **desplazamiento desde el inicio del viaje**, no la hora de reloj.

**En Excel:** `=CONTAR.SI($A$2:$A$14521;A2)>0`

Debe dar: **851 773 de 1 098 717 filas: el 77,5 %**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Agrupar `arrival_time` por su hora sin esta comprobación da **635 574 llegadas a medianoche** y un gráfico con un pico absurdo a las 00:00. El pico es el `0:00:00` de cada primera parada de cada expedición de frecuencia.


In [ ]:
p06.diagnosticar_horas(tablas)


### Paso 5 · El relleno de ceros miente al ordenar


La misma expedición mezcla `0:00:00` con una cifra y `00:00:51` con dos. Como texto, la primera es mayor que la segunda.

> ⚠️ No produce un error: produce la **primera parada de cada expedición ordenada al final**, de modo que el origen y el destino quedan cambiados. Un informe de tiempos de viaje con el signo invertido.


In [ ]:
'0:00:00' > '00:00:51'


### Paso 6 · Convertir a segundos, admitiendo horas de 24:00 y más


Un día de servicio puede pasar de la medianoche: `25:10:00` es la 01:10 del día siguiente. Es un valor legítimo de la especificación que `pd.to_datetime` rechaza.

**En Excel:** `=RESIDUO(ENTERO(E2/3600);24)`

Debe dar: **25:10:00 → 90 600 s → hora de reloj 1**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ `pd.to_datetime('25:10:00')` **falla**, y con `errors='coerce'` devuelve nulo en silencio. Las horas del servicio nocturno desaparecerían del conteo sin que nada avise.


In [ ]:
ejemplos = pd.Series(['0:00:00', '00:00:51', '25:10:00', '28:00:00'])
pd.DataFrame({'texto': ejemplos,
              'segundos': parse_hms_seconds(ejemplos),
              'hora_de_reloj': service_day_hour(parse_hms_seconds(ejemplos))})


### Paso 7 · Una ventana de frecuencia son muchas salidas


Cada fila de `frequencies.txt` dice «entre A y B sale un vehículo cada N segundos». Son 14 520 filas que representan 142 366 salidas reales.

**En Excel:** `=ENTERO((C2-B2)*86400/D2)`

Debe dar: **142 366 salidas en total**

**En Google Sheets:** la misma fórmula, sin cambios.


In [ ]:
expandido = p06.expandir_frecuencias(tablas)
expandido.head(4)


### Paso 8 · El fin de la ventana es exclusivo


La salida de `end_time` pertenece a la ventana siguiente. Es lo que fija la especificación, y de ello depende un 10 % del total.

**En Excel:** `=SUMA(E2:E14521)`

Debe dar: **64 011 correctas · 6 641 salidas de más si se cuenta inclusivo**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Es el error más fácil de defender y el más fácil de cometer: contar «de las 6 a las 7 cada 10 minutos» como siete salidas en lugar de seis. Infla el total un **10,37 %**, que es una magnitud que nadie cuestiona en un informe.


In [ ]:
comparacion = p06.comparar_ingenuo_vs_correcto(tablas)
comparacion


### Paso 9 · Expediciones por hora del día


La pregunta, contestada. Las salidas expandidas más las de horario absoluto, agrupadas por hora del día de servicio.

**En Excel:** `=CONTAR.SI($F$2:$F$14521;G2)`

Debe dar: **64 011 expediciones en el día laboral**

**En Google Sheets:** la misma fórmula, sin cambios.


In [ ]:
por_hora = p06.expediciones_por_hora(tablas)
por_hora


### Paso 10 · «El 32 % de las paradas no se usa»: el defecto que no existe


5 991 de las 18 486 filas de `stops.txt` no aparecen en la tabla de horarios. Antes de reportarlo, mirar **qué son** esas filas.

**En Excel:** `=CONTAR.SI(horarios!$D:$D;A2)=0`

Debe dar: **5 991 sin uso, de las cuales solo 2 son paradas de embarque**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Reportar «el 32 % de las paradas está huérfano» cuesta credibilidad: 5 989 de esas filas **no son paradas**, son estaciones, accesos y nodos de pasillo, y la especificación las distingue con un campo. El feed es **complejo**, no sucio.

Un informe de calidad que no mira la jerarquía del formato produce hallazgos que el publicador descarta en treinta segundos, y con ellos descarta el resto del informe.


In [ ]:
p06.paradas_sin_uso(tablas)


## 4 · El resultado

**64 011 expediciones** en el día laboral, validadas por dos caminos sin código común: expansión vectorizada con agrupación, y suma aritmética por ventana más el horario absoluto. Los dos dan la misma cifra.

Lo que habrían dado los atajos: contar expediciones declaradas, 7 905 —un **−87,65 %**—; tomar el fin de ventana como inclusivo, 70 652 —un **+10,37 %**—; agrupar `arrival_time` como hora, un pico de 635 574 llegadas a medianoche.

Y dos paradas realmente huérfanas, no 5 991.

## 5 · Qué queda aprendido

1. Un campo que se llama `arrival_time` **no siempre es una hora**. En GTFS depende de si la expedición se publica con horario fijo o con frecuencia, y hay que comprobarlo antes de agrupar por él.
2. Las horas de un día de servicio **pasan de 24:00**: `25:10:00` es la 01:10 del día siguiente y es un valor legítimo que `pd.to_datetime` rechaza.
3. Ordenar horas **como texto** pone `0:00:00` después de `00:00:51`. El relleno de ceros irregular es invisible hasta que algo sale en el orden equivocado.
4. Un intervalo publicado como «cada N minutos entre A y B» tiene el **fin exclusivo**. Contarlo inclusivo infla el total en una salida por ventana, y aquí hay 14 520.
5. `usecols` y `dtype=str` no son estilo cuando la tabla tiene 1,1 millones de filas: deciden si el análisis corre en esta máquina.
6. Leer del flujo comprimido evita duplicar 70 MB en disco sin ganar nada a cambio.
7. Un recuento que **no se puede validar por dos caminos independientes** es una afirmación, no una medición. Aquí los dos caminos dan 64 011.
8. «El 32 % de las paradas está huérfano» es un defecto que **no existe**: 5 989 de esas filas no son paradas. Reportarlo costaría más que no reportar nada.
